# Colab 25 — Pooling ablation: does `AdaptiveAvgPool` fix position-shift brittleness in retrieval?

**The question, directly.** Did adding `AdaptiveAvgPool1d(K=16)` to the encoder make true
high-similarity partners *easier to retrieve* — or was the colab15→16 jump driven by something
else? colab16 changed **two** things at once (pooling **and** band-MSE→cross-entropy), so the
famous `4oo1I01` rescue (rank 2967/1477 → 1/1) cannot, on its own, be attributed to pooling.

**Clean ablation (this notebook).** Hold *everything* fixed — training data, data order, CE
objective, epochs, seeds, eval set, pool, oracle — and toggle **only** the pooling layer:

| variant | encoder body | property |
|---|---|---|
| **A — no pooling** | `Conv → Conv → (mask) → Flatten → Linear(64·200 → 128)` | position-**rigid** FC: a 4-char N-terminal insert shifts every feature into different FC slots (the pre-colab16 architecture) |
| **B — AdaptiveAvgPool** | `Conv → Conv → (mask) → AdaptiveAvgPool1d(16) → Flatten → Linear(64·16 → 128)` | small shifts stay inside one of 16 buckets (the current architecture) |

Both variants use the **identical** 3-bin CE head and identical training pairs per seed — so any
retrieval difference is the **pooling layer alone**, isolated from the loss change.

**Grid (full, per Melissa).** `{AA-encoder, SS-encoder} × {no-pool, AvgPool} × 5 seeds` = **20
trainings**, each evaluated on **all three feeds** (AA / SS / 3Di). AA is the headline; SS/3Di are
the curiosity panel.

---

## Why AA uses *partner rank* but SS/3Di use *MAP@10* — the ill-posedness point

A retrieval task ("put query `q`'s designated partner in the top 10") is **well-posed** only when
that partner is genuinely among `q`'s top-10 nearest neighbours *in the exact-Levenshtein ground
truth itself*. It is **ill-posed** when the ground truth already places many other pool proteins at
least as close — then *no* method, not even the exact O(n²) algorithm, can rank that partner top-10.

| feed | background normLev (unrelated strings) | a "high" pair at ≥0.70 | median competitors (exact oracle) | oracle hits@10 |
|---|---|---|---|---|
| **AA** (20 letters) | ~0.15 | 0.80 is *wildly* distinctive | **0** | **100%** (rank 1) |
| **SS** (3 letters)  | ~0.5 (accidental matches) | 0.72 is *not* special | **~340** | **9%** |

So in SS the labelled partner is genuinely the ~340th-nearest neighbour — asking for it at rank ≤10
is asking for something the ground truth does not support (this is the §6 / `cross_rep.md` finding).

**MAP@10 against the exact-Levenshtein neighbour *set*** fixes this. It reframes the question from
"find *the* labelled partner" (ill-posed when 340 answers tie) to "are your top-10 *all genuine*
high-sim neighbours, ranked well?" — always well-posed. Because `AP@k` normalizes by `min(|T|,k)`,
a perfect Levenshtein-approximator scores **1.0 regardless of crowding**, so the AA-vs-SS asymmetry
no longer punishes SS mechanically.

- **AA panel** → designated-**partner rank** (valid because |T|≈1; the partner *is* the answer) → the
  `4oo1I01` dumbbell.
- **SS / 3Di panel** → **MAP@10 vs the oracle set** (the fair metric under crowding).

**Honest expectation:** SS/3Di have low oracle ceilings (24% / 33% @10), so there is little headroom
— pooling may barely move them. That is itself a clean finding: *pooling is an AA position-shift fix;
it does not help where the task is intrinsically crowded.*

**Caveat baked in:** only **5** high-AA pairs exist in CATH (10 directed queries) → AA hit@10 is
underpowered; the 5-seed mean±std is the honest read. SS (n≈606 pairs) and 3Di are powered.


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_3di.csv.gz', 'cath_eval.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab24e / colab19 / colab17b) + ablation knobs

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70
BIN_NAMES = ['far', 'mid', 'high']

K_RETRIEVAL = 10

# --- pooling-ablation knobs (the ONLY thing that differs between the two model variants) ---
POOLINGS   = ['flatten', 'avgpool']        # A = no pooling (position-rigid), B = AdaptiveAvgPool(K)
AA_SEEDS   = [42, 43, 44, 45, 46]          # 5 seeds for the AA-encoder grid
SS_SEEDS   = [52, 53, 54, 55, 56]          # 5 seeds for the SS-encoder grid
REP_SEED   = 42                            # representative seed for the per-query dumbbell figure
FOURO      = frozenset({'4oo1I01', '4ifdI01'})   # the historical position-shift smoking-gun pair
POOL_LABEL = {'flatten': 'no pooling (Flatten+Linear)', 'avgpool': f'AdaptiveAvgPool(K={K})'}

print(f'AA band_low={BAND_LOW_AA} | SS band_low={BAND_LOW_SS} | high cutoff={BAND_HIGH} '
      f'| @k={K_RETRIEVAL}')
print(f'GRID: {{AA-enc, SS-enc}} x {POOLINGS} x {len(AA_SEEDS)} seeds = '
      f'{2*len(POOLINGS)*len(AA_SEEDS)} trainings; feeds = AA/SS/3Di')

## 3. Helpers — Levenshtein, encode, perturb (RNG threaded for determinism)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) > max_len:
        raise ValueError(f'sequence length {len(seq)} exceeds max_len={max_len}')
    idx = [CHAR_TO_IDX[c] for c in seq]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); choices = [c for c in abc if c != s[i]]; s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

## 4. Architecture — the pooling-ablatable encoder

`SiameseEncoder(pooling=...)` is the **only** thing that differs between variant A and B. Everything
downstream (the CE head, the training loop, the data) is shared. For a fixed seed the two variants
see **identical training pairs and identical shuffle order** (a dedicated `DataLoader` generator
decouples shuffling from weight init), so the comparison isolates the pooling layer.

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, pooling, K=K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX, max_len=MAX_LEN):
        super().__init__()
        assert pooling in ('avgpool', 'flatten'), pooling
        self.pooling = pooling; self.K = K; self.pad_idx = pad_idx; self.max_len = max_len
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        if pooling == 'avgpool':
            self.pool = nn.AdaptiveAvgPool1d(K)
            self.fc   = nn.Linear(hidden2 * K, out_dim)           # 64*16  = 1,024  -> 128
        else:  # 'flatten' = position-rigid (the pre-colab16 architecture)
            self.pool = None
            self.fc   = nn.Linear(hidden2 * max_len, out_dim)     # 64*200 = 12,800 -> 128

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)                                  # zero out PAD positions
        if self.pooling == 'avgpool':
            h = self.pool(h).flatten(1)                           # 16 length-buckets (shift-absorbing)
        else:
            h = h.flatten(1)                                      # position-specific FC slots (rigid)
        return F.normalize(self.fc(h), p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, pooling, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(pooling)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)


class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]


# cache the per-(alphabet, seed) training pairs so BOTH pooling variants train on identical data
_PAIR_CACHE = {}
def get_pairs(alphabet, seed):
    key = (alphabet, seed)
    if key not in _PAIR_CACHE:
        _PAIR_CACHE[key] = make_full_range_pairs(alphabet, N_TRAIN, np.random.default_rng(seed))
    return _PAIR_CACHE[key]


def train_encoder(alphabet, band_low, pooling, label, seed, n_epochs=NUM_EPOCHS, verbose=False):
    pairs = get_pairs(alphabet, seed)                 # identical across pooling variants at this seed
    torch.manual_seed(seed)                           # weight init (shapes differ by architecture)
    gen = torch.Generator().manual_seed(seed)         # shuffle order decoupled from init
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    model = SiameseClassifier(pooling).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f'    [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Per-representation pools (genuinely per-feed, identical to colab24e)

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
raw = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))

RESCUED = {'4z0mC02', '3qkaE02'}

def _valid(seq, is_std, domain):
    return (isinstance(seq, str) and is_std(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or domain in RESCUED))

id_to_aa  = {d: s for d, s in zip(raw['domain_id'],   raw['aa_seq'])            if _valid(s, is_standard_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'],   raw['ss_seq'])            if _valid(s, is_standard_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_standard_aa, d)}

POOL_IDS  = {'AA': list(id_to_aa), 'SS': list(id_to_ss), '3Di': list(id_to_3di)}
LOOKUP    = {'AA': id_to_aa,       'SS': id_to_ss,       '3Di': id_to_3di}
SCORE_COL = {'AA': 'aa_score',     'SS': 'ss_score',     '3Di': '3di_score'}
POOL_SET  = {f: set(POOL_IDS[f]) for f in POOL_IDS}
FEEDS     = ['AA', 'SS', '3Di']

for f in FEEDS:
    print(f'  {f:<4} pool = {len(POOL_IDS[f]):>6}')

## 6. Feed-matched eval tables (one metric everywhere; identical protocol to colab24e)

In [ ]:
# --- load + combine + dedup unordered ---
PAIR_COLS = ['domain_a', 'domain_b', 'ss_score', 'aa_score', 'src4', 'src5', 'src_flag']
samp = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'), header=None, names=PAIR_COLS)
high = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'), header=None, names=PAIR_COLS)
src_pairs = pd.concat([samp, high], ignore_index=True)
src_pairs = src_pairs[src_pairs['domain_a'] != src_pairs['domain_b']]
src_pairs['pair_key'] = [frozenset((a, b)) for a, b in zip(src_pairs['domain_a'], src_pairs['domain_b'])]
src_pairs = src_pairs.drop_duplicates('pair_key').reset_index(drop=True)
print(f'source pairs (unordered-deduped) = {len(src_pairs):,}')

def build_feed_eval(feed):
    inpool = POOL_SET[feed]; lk = LOOKUP[feed]
    sub = src_pairs[src_pairs['domain_a'].isin(inpool) & src_pairs['domain_b'].isin(inpool)]
    a = sub['domain_a'].values; b = sub['domain_b'].values
    current = np.array([norm_lev(lk[x], lk[y]) for x, y in zip(a, b)])
    out = pd.DataFrame({'domain_a': a, 'domain_b': b, 'current_normlev': current})
    out['is_high'] = (out['current_normlev'] >= BAND_HIGH).astype(int)
    return out.reset_index(drop=True)

EVAL = {feed: build_feed_eval(feed) for feed in FEEDS}

def feed_high(feed):
    e = EVAL[feed]
    return e[e['is_high'] == 1]

def feed_queries(feed):
    h = feed_high(feed)
    return sorted(set(h['domain_a']) | set(h['domain_b']))

QUERIES = {feed: feed_queries(feed) for feed in FEEDS}
print(f'{"feed":<6}{"eligible":>10}{"high>=0.70":>12}{"queries":>9}{"pool":>9}')
for feed in FEEDS:
    print(f'{feed:<6}{len(EVAL[feed]):>10,}{int(feed_high(feed).shape[0]):>12,}'
          f'{len(QUERIES[feed]):>9,}{len(POOL_IDS[feed]):>9,}')
print('\nScope: counts are over the supplied pair-source SAMPLE after per-feed filtering '
      '(not all C(14907,2) CATH pairs). AA high pairs are genuinely scarce.')

## 7. Oracle — full-pool exact-Levenshtein neighbour sets (identical to colab24e)

In [ ]:
def build_oracle_cache(feed):
    lk = LOOKUP[feed]; pool_ids = POOL_IDS[feed]
    idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]
    lens = np.array([len(s) for s in pool_seqs])
    q_ids = QUERIES[feed]
    if not q_ids:
        return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': [], 'true_sets': {}}
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs,
                 scorer=RFLevenshtein.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]
        denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    nt = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed:<3}]: {len(q_ids):>4} queries, median |T|={int(np.median(nt))}, max={max(nt)}')
    return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': q_ids, 'true_sets': true_sets}

print('Building exact-Levenshtein oracle neighbour sets (feed-matched queries)... (slow once)')
ORACLE = {feed: build_oracle_cache(feed) for feed in FEEDS}
for feed in FEEDS:
    ts = ORACLE[feed]['true_sets']
    empty = [q for q in ORACLE[feed]['q_ids'] if len(ts[q]) == 0]
    assert not empty, f'{feed}: {len(empty)} queries with empty oracle set'
print('All feed-matched queries have >= 1 oracle neighbour  OK')

## 8. Metric machinery — oracle MAP@10 (SS/3Di) + designated-partner rank (AA)

- `mean_metrics` — MAP@10 / hit@10 / recall@50 against the exact-Levenshtein neighbour **set**
  (the well-posed, crowding-fair metric; averaged across seeds gives the error bars).
- `designated_partner_ranks` — for AA only: the rank of each query's *specific* labelled partner in
  the full pool (the historical `4oo1I01` 2967→1 number, and the dumbbell's y-value).

In [ ]:
K_LIST = (10, 50); K_MAP = 10

@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval(); valid = [d for d in ids if d in seq_lookup]; out = []
    for i in range(0, len(valid), batch_size):
        x = torch.stack([encode_pad(seq_lookup[d]) for d in valid[i:i+batch_size]]).to(device)
        out.append(model.encoder(x))
    return (torch.cat(out, 0) if out else torch.empty(0)), valid

@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def auroc_cell(model, feed):
    lk = LOOKUP[feed]; sub = EVAL[feed]; y = sub['is_high'].values
    if y.sum() == 0 or y.sum() == len(y):
        return np.nan, int(y.sum()), len(y)
    pred = pred_sim_strings(model, [lk[d] for d in sub['domain_a']], [lk[d] for d in sub['domain_b']])
    return float(roc_auc_score(y, pred)), int(y.sum()), len(y)

def _ap_at_k(order, true_set, k):
    nt = len(true_set)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for i, oi in enumerate(order[:k], start=1):
        if oi in true_set:
            hits += 1; ap += hits / i
    return ap / min(nt, k)

def mean_metrics(sim_fn, cache, k_list=K_LIST, k_map=K_MAP):
    idx = cache['idx']; ap=[]; rec50=[]; hit10=[]; nt_all=[]
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts)
        if nt == 0: continue
        sim = np.asarray(sim_fn(qi), dtype=float).copy(); sim[qi] = -np.inf
        order = np.argsort(-sim)
        nt_all.append(nt)
        ap.append(_ap_at_k(order, ts, k_map))
        ret50 = len(set(order[:50].tolist()) & ts); rec50.append(ret50/nt)
        ret10 = len(set(order[:10].tolist()) & ts); hit10.append(1.0 if ret10>=1 else 0.0)
    n = len(nt_all)
    return {'n_q': n, 'median_n_true': float(np.median(nt_all)) if n else np.nan,
            'MAP@10': float(np.mean(ap)) if n else np.nan,
            'hitrate@10': float(np.mean(hit10)) if n else np.nan,
            'recall@50': float(np.mean(rec50)) if n else np.nan}

def encoder_pool_np(model, feed):
    pe, _ = encode_pool(model, LOOKUP[feed], ORACLE[feed]['pool_ids'])
    return pe.cpu().numpy()

def designated_partner_ranks(model, feed):
    # Rank of each query's SPECIFIC labelled partner in the full pool (L2-sim). AA-only metric.
    emb = encoder_pool_np(model, feed)
    pos = {d: i for i, d in enumerate(ORACLE[feed]['pool_ids'])}
    rows = []
    for a, b in zip(feed_high(feed)['domain_a'], feed_high(feed)['domain_b']):
        if a not in pos or b not in pos: continue
        for q, p in ((a, b), (b, a)):
            qi = pos[q]; sim = 1.0 - np.linalg.norm(emb - emb[qi], axis=1) / 2.0; sim[qi] = -np.inf
            rank = int((sim > sim[pos[p]]).sum()) + 1     # 1 + #pool items strictly closer than partner
            rows.append({'query': q, 'partner': p, 'rank': rank,
                         'is_4oo1': frozenset((q, p)) == FOURO})
    return pd.DataFrame(rows)

def length_only_map(feed):
    plen = np.array([len(LOOKUP[feed][d]) for d in ORACLE[feed]['pool_ids']], dtype=float)
    return mean_metrics(lambda qi: 1.0 - np.abs(plen - plen[qi]) / np.maximum(plen, plen[qi]),
                        ORACLE[feed])['MAP@10']

LEN_MAP = {feed: length_only_map(feed) for feed in FEEDS}
print('length-only baseline MAP@10:', {f: round(v, 3) for f, v in LEN_MAP.items()})

## 9. Train the full grid + evaluate — `{AA,SS} × {no-pool,AvgPool} × 5 seeds`

20 trainings (tiny encoder, ~seconds each on GPU). Each model is evaluated on all three feeds, then
discarded to keep memory flat. AA designated-partner ranks are stored per (pooling, seed) for the
dumbbell + AA table.

In [ ]:
ENC_SPECS = [('AA-enc', AA_ALPHABET, BAND_LOW_AA, AA_SEEDS),
             ('SS-enc', SS_ALPHABET, BAND_LOW_SS, SS_SEEDS)]
IN_DOMAIN = {('AA-enc', 'AA'), ('SS-enc', 'SS')}

grid_rows = []
aa_partner_ranks = {}        # (pooling, seed) -> DataFrame of designated-partner ranks (AA-enc x AA)

for enc_label, alphabet, band_low, seeds in ENC_SPECS:
    for pooling in POOLINGS:
        for seed in seeds:
            tag = f'{enc_label}/{pooling}/seed{seed}'
            model = train_encoder(alphabet, band_low, pooling, tag, seed)
            for feed in FEEDS:
                auroc, n_pos, n_tot = auroc_cell(model, feed)
                emb = encoder_pool_np(model, feed)
                s = mean_metrics(lambda qi, e=emb: 1.0 - np.linalg.norm(e - e[qi], axis=1) / 2.0,
                                 ORACLE[feed])
                grid_rows.append({
                    'encoder': enc_label, 'pooling': pooling, 'seed': seed, 'feed': feed,
                    'role': 'in-domain' if (enc_label, feed) in IN_DOMAIN else 'cross-rep',
                    'auroc': auroc, 'n_pos': n_pos,
                    'MAP@10': s['MAP@10'], 'hitrate@10': s['hitrate@10'],
                    'recall@50': s['recall@50'], 'median_n_true': s['median_n_true'], 'n_q': s['n_q']})
            if enc_label == 'AA-enc':
                aa_partner_ranks[(pooling, seed)] = designated_partner_ranks(model, 'AA')
            print(f'  done {tag:<26} | '
                  + ' | '.join(f'{f}:MAP@10={next(r["MAP@10"] for r in grid_rows if r["encoder"]==enc_label and r["pooling"]==pooling and r["seed"]==seed and r["feed"]==f):.3f}'
                               for f in FEEDS))
            del model
            if device.type == 'cuda': torch.cuda.empty_cache()

grid_df = pd.DataFrame(grid_rows)
print(f'\ngrid rows: {len(grid_df)}  (expect {2*len(POOLINGS)*len(AA_SEEDS)*len(FEEDS)})')

## 10. Aggregate across seeds (mean ± std per encoder × pooling × feed)

In [ ]:
def agg_cell(enc, pooling, feed, metric):
    v = grid_df[(grid_df.encoder==enc) & (grid_df.pooling==pooling) & (grid_df.feed==feed)][metric]
    return float(v.mean()), float(v.std(ddof=0))

agg = (grid_df.groupby(['encoder', 'pooling', 'feed'])
       .agg(MAP_mean=('MAP@10','mean'), MAP_std=('MAP@10','std'),
            hit_mean=('hitrate@10','mean'), hit_std=('hitrate@10','std'),
            auroc_mean=('auroc','mean'), rec50_mean=('recall@50','mean'),
            med_T=('median_n_true','first'), n_q=('n_q','first'))
       .reset_index())
agg['role'] = ['in-domain' if (e, f) in IN_DOMAIN else 'cross-rep'
               for e, f in zip(agg['encoder'], agg['feed'])]

print('=' * 120)
print('FULL GRID — oracle-set retrieval, mean +/- std over 5 seeds')
print('=' * 120)
print(f'{"encoder":<9}{"pooling":<10}{"feed":<6}{"role":<11}{"MAP@10 (mean+/-sd)":>22}'
      f'{"hit@10":>16}{"AUROC":>8}{"med|T|":>8}{"n_q":>6}')
print('-' * 120)
for _, r in agg.sort_values(['encoder','feed','pooling']).iterrows():
    print(f'{r.encoder:<9}{r.pooling:<10}{r.feed:<6}{r.role:<11}'
          f'{f"{r.MAP_mean:.3f} +/- {r.MAP_std:.3f}":>22}'
          f'{f"{r.hit_mean:.3f} +/- {r.hit_std:.3f}":>16}{r.auroc_mean:>8.3f}{r.med_T:>8.0f}{r.n_q:>6.0f}')
grid_df.to_csv('colab25_grid_raw.csv', index=False)
agg.to_csv('colab25_grid_agg.csv', index=False)
print('\nsaved: colab25_grid_raw.csv, colab25_grid_agg.csv')

## 11. Figure 1 (HEADLINE) — AA designated-partner rank: no-pool vs AvgPool

One line per high-AA directed query (5 pairs × 2 directions). y = rank of the *true partner* in the
~10.5k pool (log scale, **lower = better**); dashed line at rank 10 (the hit@10 threshold). Green =
pooling improved the rank, red = worsened. The `4oo1I01 ↔ 4ifdI01` pair (the historical
position-shift smoking-gun) is drawn thick.

In [ ]:
import matplotlib.pyplot as plt

rf = aa_partner_ranks[('flatten', REP_SEED)].set_index(['query', 'partner'])
ra = aa_partner_ranks[('avgpool', REP_SEED)].set_index(['query', 'partner'])
joined = rf[['rank']].join(ra[['rank']], lsuffix='_flat', rsuffix='_pool')
joined['is_4oo1'] = [frozenset(k) == FOURO for k in joined.index]

fig, ax = plt.subplots(figsize=(7.5, 7))
x0, x1 = 0.0, 1.0
for (q, p), row in joined.iterrows():
    r0, r1 = row['rank_flat'], row['rank_pool']
    improved = r1 < r0; worsened = r1 > r0
    col = '#2ca02c' if improved else ('#d62728' if worsened else '#888888')
    lw  = 3.2 if row['is_4oo1'] else 1.2
    z   = 6 if row['is_4oo1'] else 3
    ax.plot([x0, x1], [r0, r1], color=col, lw=lw, alpha=0.9, zorder=z,
            marker='o', ms=(9 if row['is_4oo1'] else 5),
            markerfacecolor=col, markeredgecolor='black', markeredgewidth=0.6)
    if row['is_4oo1']:
        ax.annotate(f'{q}→{p}\n{int(r0)} → {int(r1)}', (x1, r1), (12, 0),
                    textcoords='offset points', fontsize=8, va='center', fontweight='bold',
                    color='#9467bd')

ax.axhline(10, ls='--', color='black', lw=1.2)
ax.text(0.5, 10*1.15, 'rank 10  (hit@10 threshold)', ha='center', fontsize=9)
ax.set_yscale('log')
ax.set_xlim(-0.25, 1.45); ax.set_xticks([x0, x1])
ax.set_xticklabels([f'A: {POOL_LABEL["flatten"]}', f'B: {POOL_LABEL["avgpool"]}'])
ax.set_ylabel('rank of true partner in pool   (log scale; lower = better)')
ax.set_title('Does pooling fix position-shift brittleness?\n'
             f'AA-encoder, AA feed, seed {REP_SEED} (clean ablation: only the pooling layer differs)')
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],color='#2ca02c',lw=2,label='improved by pooling'),
                   Line2D([0],[0],color='#d62728',lw=2,label='worsened by pooling'),
                   Line2D([0],[0],color='#888888',lw=2,label='unchanged')],
          loc='upper right', framealpha=0.9, fontsize=9)
ax.grid(True, which='both', alpha=0.25)
plt.tight_layout(); plt.savefig('colab25_fig1_pooling_rank_dumbbell.png', dpi=150, bbox_inches='tight'); plt.show()

n_hit_flat = int((joined['rank_flat'] <= 10).sum()); n_hit_pool = int((joined['rank_pool'] <= 10).sum())
print(f'seed {REP_SEED}: hit@10  no-pool {n_hit_flat}/{len(joined)}  ->  AvgPool {n_hit_pool}/{len(joined)}')
if joined['is_4oo1'].any():
    fo = joined[joined['is_4oo1']]
    print('4oo1I01<->4ifdI01 ranks (both directions):  no-pool',
          list(fo['rank_flat'].astype(int)), ' -> AvgPool', list(fo['rank_pool'].astype(int)))
else:
    print('NOTE: 4oo1I01<->4ifdI01 not in the feed-matched high-AA set this run '
          '(recomputed normLev may have dropped it); dumbbell still shows all high-AA queries.')

## 12. Table — AA pooling ablation (hit@10, MAP@10, median partner rank, 4oo1I01 rank)

In [ ]:
rows = []
for pooling in POOLINGS:
    # hit@10 + median rank pooled across all seeds' AA directed queries
    dfs = [aa_partner_ranks[(pooling, s)] for s in AA_SEEDS]
    per_seed_hit = [float((d['rank'] <= 10).mean()) for d in dfs]
    all_ranks = np.concatenate([d['rank'].values for d in dfs])
    # 4oo1I01 rank: representative seed, both directions
    rep = aa_partner_ranks[(pooling, REP_SEED)]
    fo = rep[rep['is_4oo1']]
    fo_str = ' / '.join(str(int(x)) for x in fo['rank']) if len(fo) else 'n/a'
    map_mean, map_std = agg_cell('AA-enc', pooling, 'AA', 'MAP@10')
    rows.append({'pooling': POOL_LABEL[pooling],
                 'hit@10 (mean+/-sd over seeds)': f'{np.mean(per_seed_hit):.2f} +/- {np.std(per_seed_hit):.2f}',
                 'MAP@10 (mean+/-sd)': f'{map_mean:.3f} +/- {map_std:.3f}',
                 'median partner rank': f'{np.median(all_ranks):.0f}',
                 '4oo1I01 rank (seed42 a->b/b->a)': fo_str})
aa_table = pd.DataFrame(rows)
print('=' * 100)
print('AA POOLING ABLATION  (AA-encoder x AA feed; designated-partner retrieval; 10 directed queries/seed)')
print('=' * 100)
with pd.option_context('display.max_colwidth', None, 'display.width', 140):
    print(aa_table.to_string(index=False))
aa_table.to_csv('colab25_aa_ablation_table.csv', index=False)
print('\nsaved: colab25_aa_ablation_table.csv')
print('Read: a large no-pool median rank / low no-pool hit@10 that AvgPool lifts = pooling is the lever '
      '(isolated from the loss, which is CE in BOTH variants).')

## 13. Figure 2 (CURIOSITY) — does pooling help SS / 3Di? Full-grid MAP@10

MAP@10 vs the exact-Levenshtein neighbour set (the fair, crowding-aware metric), mean ± std over 5
seeds. Four bars per feed: AA-encoder and SS-encoder, each in {no-pool, AvgPool}. Grey dashed line =
length-only baseline; `med|T|` = crowding. **Expectation:** pooling helps AA in-domain; SS/3Di sit
near a low oracle-bounded ceiling, so pooling barely moves them — pooling is an AA position-shift fix,
not a cure for crowded low-entropy alphabets.

In [ ]:
COL = {('AA-enc','flatten'): '#aec7e8', ('AA-enc','avgpool'): '#1f77b4',
       ('SS-enc','flatten'): '#98df8a', ('SS-enc','avgpool'): '#2ca02c'}
series = [('AA-enc','flatten'), ('AA-enc','avgpool'), ('SS-enc','flatten'), ('SS-enc','avgpool')]
labels = {('AA-enc','flatten'):'AA-enc, no-pool', ('AA-enc','avgpool'):'AA-enc, AvgPool',
          ('SS-enc','flatten'):'SS-enc, no-pool', ('SS-enc','avgpool'):'SS-enc, AvgPool'}

x = np.arange(len(FEEDS)); w = 0.19
fig, ax = plt.subplots(figsize=(10.5, 5.8))
for j, (enc, pool) in enumerate(series):
    means = [agg_cell(enc, pool, f, 'MAP@10')[0] for f in FEEDS]
    stds  = [agg_cell(enc, pool, f, 'MAP@10')[1] for f in FEEDS]
    off = (j - 1.5) * w
    ax.bar(x + off, means, w, label=labels[(enc,pool)], color=COL[(enc,pool)])
    ax.errorbar(x + off, means, yerr=stds, fmt='none', ecolor='black', capsize=2.5, lw=1)
for i, f in enumerate(FEEDS):
    ax.hlines(LEN_MAP[f], x[i]-2*w, x[i]+2*w, color='#555555', ls='--', lw=1.3,
              label='length-only baseline' if i == 0 else None)
    mt = agg[(agg.feed==f)]['med_T'].iloc[0]
    ax.annotate(f'med|T|={mt:.0f}', (i, 0), (0, -30), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel(f'MAP@{K_MAP}  (vs exact-Levenshtein neighbour set)')
ax.set_xlabel('Feed modality'); ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Pooling effect across the full grid (mean +/- std over 5 seeds)\n'
             'in-domain = AA-enc/AA and SS-enc/SS; the rest are cross-representation')
ax.legend(loc='upper right', framealpha=0.9, fontsize=8.5, ncol=2)
plt.tight_layout(); plt.savefig('colab25_fig2_grid_map.png', dpi=150, bbox_inches='tight'); plt.show()

## 14. (Optional) architecture schematic for the slide — before / after

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
def draw(ax, title, blocks, caption, accent):
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    y = 5.0
    for i, (txt, col) in enumerate(blocks):
        ax.add_patch(plt.Rectangle((1.5, y-0.45), 7, 0.7, fc=col, ec='black', lw=1.2))
        ax.text(5, y-0.1, txt, ha='center', va='center', fontsize=9.5)
        if i < len(blocks)-1:
            ax.annotate('', (5, y-0.55), (5, y-0.95), arrowprops=dict(arrowstyle='->', lw=1.3))
        y -= 1.15
    ax.text(5, 0.15, caption, ha='center', va='bottom', fontsize=8.5, style='italic', color=accent)

draw(axes[0], 'A — no pooling (position-rigid)',
     [('Embedding (21, 32)', '#f0f0f0'), ('Conv1d x2  -> (B, 64, 200)', '#f0f0f0'),
      ('Flatten -> 64x200 = 12,800', '#ffd6d6'), ('Linear(12,800 -> 128)', '#ffd6d6')],
     'a small N-terminal insertion shifts every\nfeature into a different FC slot', '#d62728')
draw(axes[1], f'B — AdaptiveAvgPool(K={K})',
     [('Embedding (21, 32)', '#f0f0f0'), ('Conv1d x2  -> (B, 64, 200)', '#f0f0f0'),
      (f'AdaptiveAvgPool1d({K}) -> 64x{K} = {64*K}', '#d6f0d6'), (f'Linear({64*K} -> 128)', '#d6f0d6')],
     'the insertion is absorbed inside one of\n16 local buckets -> features stay aligned', '#2ca02c')
plt.tight_layout(); plt.savefig('colab25_arch_schematic.png', dpi=150, bbox_inches='tight'); plt.show()

## How to read this notebook

**The clean claim.** Both variants share data, data order, CE objective, epochs, seeds, pool, and
oracle; only the pooling layer differs. So Figure 1 / the AA table attribute any retrieval change to
**pooling alone** — fixing the colab15→16 confound (which also flipped MSE→CE).

- **Figure 1 (headline, AA):** if the no-pool side strands `4oo1I01` (and others) at high rank while
  AvgPool pulls them to ~1 (below the rank-10 line), pooling *causes* the retrieval fix. This is the
  slide's right-hand graphic; §14 is the left-hand architecture visual.
- **Table (§12):** the quantitative companion — hit@10 (mean±std over seeds), MAP@10, median partner
  rank, and the `4oo1I01` rank verbatim.
- **Figure 2 (curiosity, SS/3Di):** MAP@10 against the exact-Levenshtein **set** (fair under
  crowding). Read each feed against its grey length-only baseline and `med|T|`. Expect AA in-domain to
  clear the baseline with pooling; SS/3Di to be near their low oracle ceilings regardless of pooling
  — the honest "pooling is an AA fix" message.

**Caveats.** AA high pairs are n=5 (10 directed queries) — the 5-seed spread is the honest read.
SS/3Di are powered. Relevance is the exact-Levenshtein / high string-similarity neighbour set; **no
biological claim is made**. The head is a nominal 3-bin (far/mid/high) classifier with a
near-degenerate far class (practical signal = mid-vs-high), identical in both pooling variants.